In [1]:
import os
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    default_data_collator,
)

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

In [2]:
BASE_MODEL = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
DATA_FILE = "spider_style_dataset.jsonl"   # your dataset path
OUTPUT_DIR = "qwen_spider_full_sft_v3"              # NEW name

os.makedirs(OUTPUT_DIR, exist_ok=True)

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

# needed when using gradient checkpointing
model.config.use_cache = False


`torch_dtype` is deprecated! Use `dtype` instead!


In [4]:
raw_ds = load_dataset("json", data_files=DATA_FILE, split="train")
len(raw_ds), raw_ds[0]


(2503,
 {'messages': [{'role': 'system', 'content': 'You are a text-to-SQL expert.'},
   {'role': 'user',
    'content': 'Given schema:\nCREATE TABLE journals (\n  journal_id INTEGER PRIMARY KEY,\n  name TEXT,\n  publisher TEXT,\n  issn TEXT,\n  impact_factor REAL\n);\n\nCREATE TABLE articles (\n  article_id INTEGER PRIMARY KEY,\n  title TEXT,\n  pub_date DATE,\n  journal_id INTEGER\n);\n\nCREATE TABLE authors (\n  author_id INTEGER PRIMARY KEY,\n  name TEXT,\n  affiliation TEXT,\n  country TEXT\n);\n\nCREATE TABLE article_authors (\n  aa_id INTEGER PRIMARY KEY,\n  article_id INTEGER,\n  author_id INTEGER\n);\n\nCREATE TABLE citations (\n  citation_id INTEGER PRIMARY KEY,\n  citing_article_id INTEGER,\n  cited_article_id INTEGER,\n  citation_year INTEGER\n);\n\nQuestion: What is the average impact factor of all journals?'},
   {'role': 'assistant',
    'content': 'SELECT AVG(impact_factor) FROM journals;'}]})

In [5]:
MAX_SEQ_LEN = 512

def build_prompt_and_answer(example):
    msgs = example["messages"]
    # assume final message is assistant with SQL
    assert msgs[-1]["role"] == "assistant"
    answer = msgs[-1]["content"]

    # everything before final assistant = prompt (system + user)
    prompt_messages = msgs[:-1]

    # add_generation_prompt=True so model expects to generate assistant
    prompt = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    full_text = prompt + answer

    tok_prompt = tokenizer(
        prompt,
        truncation=True,
        max_length=MAX_SEQ_LEN,
        padding="max_length",
    )
    tok_full = tokenizer(
        full_text,
        truncation=True,
        max_length=MAX_SEQ_LEN,
        padding="max_length",
    )

    input_ids = tok_full["input_ids"]
    attention_mask = tok_full["attention_mask"]

    # number of real tokens in prompt
    prompt_len = sum(tok_prompt["attention_mask"])

    labels = [-100] * MAX_SEQ_LEN
    for i in range(prompt_len, MAX_SEQ_LEN):
        if attention_mask[i] == 1:  # non-pad
            labels[i] = input_ids[i]

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }

proc_ds = raw_ds.map(
    build_prompt_and_answer,
    remove_columns=raw_ds.column_names,
    desc="Tokenizing & building labels",
)

proc_ds[0].keys(), len(proc_ds[0]["input_ids"]), len(proc_ds[0]["labels"])


(dict_keys(['input_ids', 'attention_mask', 'labels']), 512, 512)

In [6]:
split = proc_ds.train_test_split(test_size=0.1, seed=42)  # 10% val
train_ds = split["train"]
eval_ds = split["test"]

len(train_ds), len(eval_ds)


(2252, 251)

In [7]:
BATCH_SIZE = 2
GRAD_ACCUM = 32        # effective batch = 64
EPOCHS = 3
LR = 5e-6

EVAL_STEPS = 200     # eval/save every 200 optimizer steps
LOG_STEPS = 20

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    weight_decay=0.01,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",

    logging_steps=LOG_STEPS,

    eval_strategy="steps",
    eval_steps=EVAL_STEPS,

    save_strategy="steps",
    save_steps=EVAL_STEPS,
    save_total_limit=2,              # keep last 2 checkpoints

    load_best_model_at_end=True,     # <- important
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    gradient_checkpointing=True,
    report_to="none",
)

# (optional but sometimes helps)
model.gradient_checkpointing_enable()


In [8]:
data_collator = default_data_collator

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

trainer.train()

# after train, trainer.model is best-val-loss checkpoint
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)


/tmp/ipykernel_3956/1657162300.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The model is already on multiple devices. Skipping the move to device specified in `args`.


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss


('qwen_spider_full_sft_v3/tokenizer_config.json',
 'qwen_spider_full_sft_v3/special_tokens_map.json',
 'qwen_spider_full_sft_v3/chat_template.jinja',
 'qwen_spider_full_sft_v3/vocab.json',
 'qwen_spider_full_sft_v3/merges.txt',
 'qwen_spider_full_sft_v3/added_tokens.json',
 'qwen_spider_full_sft_v3/tokenizer.json')

In [9]:
ft_tokenizer = AutoTokenizer.from_pretrained(
    OUTPUT_DIR,
    trust_remote_code=True,
)
if ft_tokenizer.pad_token is None:
    ft_tokenizer.pad_token = ft_tokenizer.eos_token

ft_model = AutoModelForCausalLM.from_pretrained(
    OUTPUT_DIR,
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
ft_model.eval()

def generate_from_ft(example, max_new_tokens=256):
    msgs = example["messages"]
    # use all non-assistant messages as prompt (system + user)
    eval_msgs = [m for m in msgs if m["role"] != "assistant"]

    prompt = ft_tokenizer.apply_chat_template(
        eval_msgs,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = ft_tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        out_ids = ft_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=ft_tokenizer.pad_token_id,
            eos_token_id=ft_tokenizer.eos_token_id,
        )

    gen_ids = out_ids[0, inputs["input_ids"].shape[1]:]
    return ft_tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

print(generate_from_ft(raw_ds[0]))


The tokenizer you are loading from 'qwen_spider_full_sft_v3' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


SELECT AVG(impact_factor) FROM journals;
